# Train the paper's pretrained Hier-BiLSTM-CRF on Colab

The role classifier uses **raw 200-dimensional legal sent2vec vectors → BiLSTM (100 units per direction) → dropout 0.5 → seven role scores → CRF**. This is the original pretrained variant. The TF-IDF, S-BERT, Legal-BERT and OpenAI encoders belong to the separate summarization comparison.

Choose a hosted CPU runtime and run the cells in order. Training uses 32 judgments; eight validation judgments select the optimizer schedule and epoch. A fresh final model uses all 40 training judgments. The ten reserved judgments are not in the feature archive and are evaluated separately afterward. No API key or native sent2vec installation is needed on Colab.


In [ ]:
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import platform
import subprocess
import sys
import tempfile
import time
from urllib.request import urlretrieve
from zipfile import ZipFile, ZIP_DEFLATED
import google.colab
import torch

paper_revision = 'be4cee13542173865f37b34bb0ebba7e5cdf22f4'
repository = 'https://github.com/SrikarKorrapati/Extractive-Summarisation-of-Legal-Document.git'
if 'paper_work' not in globals():
    paper_work = Path(tempfile.mkdtemp(prefix='paper-roles-', dir='/content'))
paper_source = paper_work / 'source'
if not paper_source.exists():
    subprocess.run(['git', 'init', '--quiet', str(paper_source)], check=True)
    subprocess.run(['git', 'fetch', '--quiet', '--depth=1', repository, paper_revision], cwd=paper_source, check=True)
    subprocess.run(['git', 'checkout', '--quiet', '--detach', 'FETCH_HEAD'], cwd=paper_source, check=True)
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=paper_source, text=True).strip() == paper_revision
assert not subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=no'], cwd=paper_source)
paper_started = datetime.now(timezone.utc).isoformat()
print(paper_work, 'Google Colab CPU', platform.python_version(), 'PyTorch', torch.__version__)


In [ ]:
paper_archive = paper_work / 'paper_role_features.zip'
feature_url = 'https://github.com/SrikarKorrapati/Extractive-Summarisation-of-Legal-Document/releases/download/paper-role-features-v1/paper_role_features.zip'
feature_sha = 'ae18499a78ba2eb53c81c622951c6cecba49b7ee6bc6ce2d1c3cba55f9f5abd4'
if not paper_archive.exists():
    urlretrieve(feature_url, paper_archive)
assert hashlib.sha256(paper_archive.read_bytes()).hexdigest() == feature_sha
paper_features = paper_work / 'features'
with ZipFile(paper_archive) as archive:
    assert sorted(archive.namelist()) == ['features.npz', 'manifest.json']
    if not paper_features.exists():
        archive.extractall(paper_features)
manifest = json.loads((paper_features / 'manifest.json').read_text())
assert len(manifest['included_case_ids']) == 40
assert not set(manifest['included_case_ids']) & set(manifest['case_ids']['test'])
print('Verified raw features for 32 fitting and 8 validation judgments; test judgments excluded.')


In [ ]:
def launch_paper(stage, output, extra=()):
    if output.exists() and any(output.iterdir()):
        raise RuntimeError(f'Output already exists: {output}. Inspect it before restarting.')
    log = paper_work / (stage + '.log')
    if log.exists():
        raise RuntimeError(f'Job log already exists: {log}. Do not start a duplicate job.')
    with log.open('x') as stream:
        process = subprocess.Popen([sys.executable, '-u', '-m', 'legal_lexrank.paper_role_training',
            stage, '--data', str(paper_source), '--features', str(paper_features), '--output', str(output),
            *map(str, extra)], cwd=paper_source, stdout=stream, stderr=subprocess.STDOUT)
    return process, log

def watch(process, log):
    previous = ''
    while True:
        current = log.read_text()
        if current != previous:
            print(current[len(previous):], end='', flush=True)
            previous = current
        status = process.poll()
        if status is not None:
            if status:
                raise RuntimeError(f'Training exited with status {status}; inspect {log}')
            break
        time.sleep(15)

paper_tuning = paper_work / 'tuning'
if 'paper_process' not in globals():
    paper_process, paper_log = launch_paper('tune', paper_tuning, ['--max-epochs', '300', '--patience', '40'])
watch(paper_process, paper_log)
selection = json.loads((paper_tuning / 'selection.json').read_text())
print('Validation-selected configuration:', selection['selected']['config'])
print('Selected epoch:', selection['selected']['best_epoch'])
print('Validation accuracy:', selection['selected']['validation']['accuracy'])


In [ ]:
paper_final = paper_work / 'final'
if 'paper_final_process' not in globals():
    paper_final_process, paper_final_log = launch_paper('refit', paper_final,
        ['--selection', paper_tuning / 'selection.json'])
watch(paper_final_process, paper_final_log)
print('Fresh final classifier fitted on all 40 training judgments.')


In [ ]:
from google.colab import files
runtime = {'platform':'Google Colab', 'device':'cpu', 'python':platform.python_version(),
    'torch':str(torch.__version__), 'source_revision':paper_revision, 'started_utc':paper_started,
    'finished_utc':datetime.now(timezone.utc).isoformat(), 'feature_archive_sha256':feature_sha}
(paper_final / 'runtime.json').write_text(json.dumps(runtime, indent=2))
result = paper_work / 'paper_role_labeller.zip'
with ZipFile(result, 'w', compression=ZIP_DEFLATED) as archive:
    for path in sorted(paper_final.iterdir()):
        if path.is_file():
            archive.write(path, path.name)
    archive.write(paper_tuning / 'selection.json', 'validation/selection.json')
    for path in sorted(paper_tuning.glob('candidate-*/history.json')):
        archive.write(path, 'validation/' + str(path.relative_to(paper_tuning)))
    archive.write(paper_log, 'validation/training.log')
    archive.write(paper_final_log, 'final-training.log')
print('Archive SHA256:', hashlib.sha256(result.read_bytes()).hexdigest())
files.download(str(result))


After extracting the bundle locally, evaluate the frozen model with:

```sh
python -m legal_lexrank.evaluate_roles --data . --role-run runs/paper-colab --output runs/paper-role-evaluation
```

New-text inference needs the official native sent2vec binding and model binary described in the repository README. Do not change the chosen model or epoch based on the ten-case test results.
